# 9.2 Topic modeling with LDA

**Part 1:** a complete reference analysis. **Part 2:** repeat it with a dataset you find and interpret the results. You may reuse the supplied code. The teacher grades your analysis and reflection manually; assertions only check that required objects were created. Read the README for scoring and submission details.

LDA means **Latent Dirichlet Allocation**, not Linear Discriminant Analysis. A topic is a distribution over words; each document has a distribution over topics. Topic names are interpretations we supply, not labels the model knows.


# Part 1 — Complete reference implementation

This small, fictional English news-headline dataset is included for an offline demonstration. It contains repeated vocabulary and some mixed subjects. It is not a representative news sample. Read and execute every cell in this part; no implementation or answers are required here.


In [ ]:
from pathlib import Path
import csv
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

activity_dir = Path.cwd() if (Path.cwd() / "data/headlines.csv").exists() else Path.cwd().parent
with (activity_dir / "data/headlines.csv").open(newline="", encoding="utf-8") as file:
    reference_documents = [row["headline"] for row in csv.DictReader(file)]
print(f"Reference documents: {len(reference_documents)}")
print(*reference_documents[:3], sep="\n")


## 1. Create document–term counts

`CountVectorizer` creates one row per document and one column per vocabulary word. It lowercases text and counts tokens; `stop_words="english"` removes common English words. We use **counts**, not TF-IDF, as LDA input. The small preview below labels its columns with vocabulary words. Only that preview is converted to a dense array.


In [ ]:
reference_vectorizer = CountVectorizer(stop_words="english")
reference_counts = reference_vectorizer.fit_transform(reference_documents)
reference_terms = reference_vectorizer.get_feature_names_out()
print("Count matrix shape:", reference_counts.shape)
preview_columns = reference_counts[0].indices[:8]
print("Preview words:", reference_terms[preview_columns].tolist())
reference_counts[:3, preview_columns].toarray()


## 2. Fit LDA and display the direct output of fit_transform

Create a model with four topics, then call `fit_transform` on the count matrix. The **raw return value** is a NumPy array, displayed directly as the final expression below. Nothing has been ranked, grouped, or converted into a summary yet.

Each **row** corresponds to one document in `reference_documents`, in the original order. Each **column** corresponds to a topic ID, 0 through 3. An entry is the estimated proportion of that topic in that document. The four values in each row sum to one, apart from floating-point rounding. The model learns topic word distributions and infers document mixtures; it does not return topic names or one hard label per document.


In [ ]:
reference_model = LatentDirichletAllocation(
    n_components=4, random_state=42, learning_method="batch", max_iter=30
)
reference_document_topics = reference_model.fit_transform(reference_counts)
reference_document_topics


## 3. Connect rows of that array to documents

The shape is `(number_of_documents, number_of_topics)`. Indexing `[document_id]` selects the full topic mixture of one document; indexing `[document_id, topic_id]` selects one weight. Below, each printed row is paired with its original headline. These are model estimates, not manually verified labels.


In [ ]:
print("Document-topic shape:", reference_document_topics.shape)
print("Columns:", [f"Topic {i}" for i in range(reference_model.n_components)])
for document_id in range(3):
    print(f"Document {document_id}: {reference_documents[document_id]}")
    print(reference_document_topics[document_id].round(3))
    print("Row sum:", reference_document_topics[document_id].sum())


## Read a document's topic mixture

Choose a headline that combines subjects. Its row contains one weight per topic. The largest weight can help summarize it, but the document still has a mixture. The three examples shown for a topic are documents with high weight for that topic; the same document can appear under multiple topics. Read the actual text to decide whether that association makes sense.


In [ ]:
reference_document_id = reference_documents.index(
    "Bank invests in software security for mobile payments"
)
print(reference_document_id, reference_documents[reference_document_id])
for topic_id, weight in enumerate(reference_document_topics[reference_document_id]):
    print(f"Topic {topic_id}: {weight:.3f}")


## 4. Get the top words for each topic

The output of `fit_transform` describes **documents**, not words. For words, inspect the fitted model's `components_`: it has one row per topic and one column per vocabulary word. Its entries are positive model weights, not normalized probabilities or raw word counts. Divide each row by its sum to get an estimated word distribution for that topic.

For each topic, sort word probabilities from largest to smallest, keep eight indices, and map those indices back to `reference_terms`. Each printed probability refers to a word within a topic; it is different from a topic's proportion within a document. The displayed eight probabilities need not sum to one because the remaining words are omitted.


In [ ]:
print("Topic-word shape:", reference_model.components_.shape)
reference_topic_words = (
    reference_model.components_ / reference_model.components_.sum(axis=1, keepdims=True)
)
for topic_id, probabilities in enumerate(reference_topic_words):
    word_ids = np.argsort(-probabilities, kind="stable")[:8]
    print(f"Topic {topic_id}")
    for word_id in word_ids:
        print(f"  {reference_terms[word_id]}: {probabilities[word_id]:.3f}")


## 5. Find representative documents for each topic

Select **a column** of the document-topic array to get that topic's weight across all documents. Sort the document IDs by descending weight and read the first three original texts. A document can rank highly for more than one topic; this is not an exclusive partition into groups.

We also collect the displayed words and document IDs into `reference_summaries`, so the same information can be reused later.


In [ ]:
reference_summaries = []
for topic_id in range(reference_model.n_components):
    document_ids = np.argsort(-reference_document_topics[:, topic_id], kind="stable")[:3]
    word_ids = np.argsort(-reference_topic_words[topic_id], kind="stable")[:8]
    print(f"Topic {topic_id}")
    for document_id in document_ids:
        weight = reference_document_topics[document_id, topic_id]
        print(f"  Document {document_id}, weight {weight:.3f}: {reference_documents[document_id]}")
    reference_summaries.append({
        "topic_id": topic_id,
        "words": [(str(reference_terms[i]), float(reference_topic_words[topic_id, i])) for i in word_ids],
        "document_ids": document_ids.tolist(),
    })


## 6. Reuse these steps

The functions below package the steps we just executed: counts, LDA, document-topic distributions, top words, and representative documents. They introduce no new modeling method. You may use them directly for your own corpus in Part 2.

We first collect the already computed reference objects into the same dictionary format. The helper will be used for the second topic count; it is not needed to obtain or read the first model's `fit_transform` output.


In [ ]:
reference_results = {
    "vectorizer": reference_vectorizer,
    "counts": reference_counts,
    "terms": reference_terms,
    "model": reference_model,
    "document_topics": reference_document_topics,
    "summaries": reference_summaries,
}


In [ ]:
def analyze_topics(documents, n_topics=4, stop_words="english"):
    vectorizer = CountVectorizer(stop_words=stop_words)
    counts = vectorizer.fit_transform(documents)
    model = LatentDirichletAllocation(
        n_components=n_topics, random_state=42, learning_method="batch", max_iter=30
    )
    document_topics = model.fit_transform(counts)
    terms = vectorizer.get_feature_names_out()
    topic_words = model.components_ / model.components_.sum(axis=1, keepdims=True)
    summaries = []
    for topic_id, probabilities in enumerate(topic_words):
        word_ids = np.argsort(-probabilities, kind="stable")[:8]
        document_ids = np.argsort(-document_topics[:, topic_id], kind="stable")[:3]
        summaries.append({
            "topic_id": topic_id,
            "words": [(str(terms[i]), float(probabilities[i])) for i in word_ids],
            "document_ids": document_ids.tolist(),
        })
    return {
        "vectorizer": vectorizer, "counts": counts, "terms": terms,
        "model": model, "document_topics": document_topics, "summaries": summaries,
    }


def show_topics(results, documents):
    print(f"Documents: {len(documents)}; topics: {results['model'].n_components}")
    for summary in results["summaries"]:
        topic_id = summary["topic_id"]
        words = ", ".join(f"{word} ({weight:.3f})" for word, weight in summary["words"])
        print(f"\nTopic {topic_id}: {words}")
        for document_id in summary["document_ids"]:
            weight = results["document_topics"][document_id, topic_id]
            excerpt = " ".join(documents[document_id].split())[:240]
            print(f"  Document {document_id}, topic weight {weight:.2f}: {excerpt}")


## Change only the topic count

We repeat the analysis with three topics, keeping the documents, preprocessing, and random seed the same. Compare words and documents to find themes that persist, merge, or split. Topic 0 in one model does not necessarily mean the same thing as topic 0 in another. A larger count does not guarantee a better interpretation.


In [ ]:
reference_alternative = analyze_topics(reference_documents, n_topics=3)
show_topics(reference_alternative, reference_documents)


## Worked interpretation

These are tentative names for the four-topic run, based on the printed words and documents. Notice the contradictions: choosing four topics does not recover four perfectly separated subjects.

| Topic ID | Tentative name | Evidence and caveat |
|---|---|---|
| 0 | Healthcare, with technology mixed in | `hospital`, `vaccine`, `doctors`, and `clinic` suggest healthcare. Documents 36 (nurses and patient care) and 50 (rural vaccine access) support that name. But document 18 is about software security and artificial intelligence, so this is not a pure health topic. |
| 1 | Football and technology, mixed | `football`, `league`, and `team` suggest sport, supported by document 33 (a football final). Yet documents 13 and 60 concern artificial intelligence tools, matching `software`, `company`, and `artificial`. Calling the whole topic only "football" would hide that conflict. |
| 2 | Economy and banking, with software | `bank`, `inflation`, `growth`, and `consumer` suggest economic news. Documents 44 (growth despite inflation) and 41 (rising food prices) support this interpretation. The software terms show that the topic has other associations too. |
| 3 | Business finance, with sport mixed in | `bank`, `loans`, `business`, and `investment` match documents 46 and 29 about business loans. However, document 59 is about football training technology, and `players` and `football` appear among the leading words. This is another uncertain boundary. |

**Ambiguous document:** document 58, "Bank invests in software security for mobile payments", genuinely combines banking and technology. Its estimated mixture is approximately 0.037, 0.036, 0.890, and 0.036. The large weight on topic 2 is plausible because that topic includes both `bank` and software-related words. Still, the mixture concentrates almost everything in one topic rather than clearly separating the two subjects. Model confidence is not proof of a uniquely correct interpretation.

**Three versus four topics:** the three-topic run still has a healthcare-oriented topic (0), a mixed football/technology topic (1), and an economy-oriented topic (2). Documents 36 and 50 still support healthcare, while documents 44 and 41 still support economic news. Document 59 moves from topic 3 in the four-topic model to topic 1 in the three-topic model. Its content, rather than its numerical ID, lets us understand that change. Three topics provide a shorter overview; four expose an additional, but mixed, business-finance grouping. Neither count removes the technology overlap, so I would describe the three-topic model as a simpler overview rather than claim it is quantitatively better.

**Limitations:** these synthetic headlines are short, use repeated vocabulary, and contain little context. Their selection favors some words and subjects. English stopword removal is convenient but cannot resolve ambiguous meanings. Counts ignore word order; high-weight words and strongly associated documents can still form an incoherent theme. These observations describe this small collection, not the distribution of topics in real news.


# Part 2 — Your dataset and interpretation

Find a different text dataset and repeat the reference workflow. Any subject or language is acceptable. Prefer substantive documents with varied vocabulary; about 100–1,000 documents is a practical starting point, not a graded minimum. Reuse or adapt the helpers: implementation style is not scored.

Add your own code and markdown cells after the prompts, before the corresponding completion checks. Preserve all supplied cells and IDs. No answers or student analysis are provided in this part. All 100 points are awarded by teacher review; passing checks only establishes that the main objects exist.


## A. Choose and load the data — Q1 (15 points)

Find a dataset other than the reference corpus. Create `student_documents`, a list of nonempty document strings. Keep loading code reproducible and paths relative to `activity_dir`. Record the original source link, language, document unit, number of documents, selection or sampling, and preprocessing. Provide accessible input files or exact download and loading instructions; the teacher must be able to obtain the same documents without a private account. Do not commit large raw datasets or credentials.

**Reflection Q1:** Where did the texts come from, what does each document represent, and what did you hope to discover? Why is this collection suitable? Describe selection, language, preprocessing, size, and access for reproduction.

Add a code cell for loading and a markdown cell with your answer below. For a CSV, adapt the reference loading code and its column name.


In [ ]:
# Completion only: no fixed dataset size, source, or content is tested.
assert "student_documents" in globals(), "Create student_documents in a new code cell."
assert len(student_documents) > 0, "Load your chosen text documents."


## B. Run and inspect LDA — completed analysis (15 points across B and D)

Choose at least two topics. Create `student_results` by calling `analyze_topics(student_documents, n_topics=..., stop_words=...)`, then call `show_topics(student_results, student_documents)`. Record your settings and explain preprocessing choices. The helper uses English stopwords by default: for another language, supply a suitable list or `None`. Discard blank texts and describe that decision. You may adapt the helper while preserving the result keys.

Print the topic mixture for a document you want to discuss. Follow the reference mixture cell, using your own document ID. Read full documents when the displayed excerpts are insufficient. Reusing code is encouraged; there are no required topic names or model-quality thresholds.

Add your analysis code below.


In [ ]:
# Completion only: no algorithm behavior or expected numerical values are tested.
assert "student_results" in globals(), "Create student_results."
assert student_results.get("model") is not None, "Create the LDA model."
assert student_results.get("document_topics") is not None, "Create document-topic results."
assert student_results.get("summaries"), "Create topic summaries."


## C. Interpret the topics — Q2 (35 points) and Q3 (10 points)

**Reflection Q2:** What name would you give **each** topic in your primary model? For every name, cite high-weight terms and at least two document IDs or excerpts. Explain why the evidence supports the name. Discuss contradictions instead of hiding them; "mixed" or "unclear" can be justified names.

**Reflection Q3:** Which topic is hardest to interpret? Discuss a document whose topic mixture is ambiguous or surprising. Cite its topic weights and explain whether its content supports that mixture. It is also valid to explain an unexpected, overly confident association.

Add your answers in new markdown cells. There is no required response length or single correct list of names. The teacher reviews evidence and reasoning, not just whether an answer exists.


## D. Compare one different count — Q4 (10 points)

Create `alternative_results` by repeating LDA on the same `student_documents` with a different topic count of at least two, keeping preprocessing and the random seed unchanged. Display its topic words and representative documents. You do not need to search many counts or calculate evaluation metrics.

**Reflection Q4:** What changed? Identify themes that split, merge, persist, or become harder to interpret, citing terms and documents from both runs. Match themes by content, not topic IDs. Which result is more useful for your exploration question, and why?

Add the alternative analysis in a new code cell and your answer in a new markdown cell below. The 15 completed-analysis points cover readable results from both runs, including topic mixtures.


In [ ]:
# Completion only: the teacher reviews the comparison and modeling choices.
assert "alternative_results" in globals(), "Create alternative_results."
assert alternative_results.get("model") is not None, "Create the alternative model."
assert alternative_results.get("document_topics") is not None, "Create its topic mixtures."
assert alternative_results.get("summaries"), "Create its topic summaries."


## E. Conclusions and limitations — Q5 (15 points)

**Reflection Q5:** What did you learn about this corpus? How might preprocessing, document length, dataset selection, and the model's word-count representation affect your conclusions? Distinguish observations supported by these texts from claims about the wider world.

Add your answer in a new markdown cell. The teacher manually reviews all five answers, the source and reproducibility, and the model outputs. There are no assertions on the content or length of your reflection answers.

Before submitting, restart and run the whole notebook. Clear saved outputs with the README commands, retaining your code, answers, source links, and data access instructions. The teacher can rerun your notebook to inspect results. No train/test split, accuracy, coherence, perplexity, LSI implementation, or comparison between algorithms is required.
